# Pipeline demo: MegaDetector v6 → BioCLIP-2

Minimal end-to-end run on a small image folder. Same code path as the `wytrap detect` CLI — use this notebook for exploration and the CLI for SLURM jobs.

**Setup**: install wytrap once into your environment:

```bash
pip install -e ../wytrap
```

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from wytrap.run import process_folder
from wytrap.io import load_record

In [2]:
# Edit these two paths.
INPUT_DIR  = "/home/omartin9/wildimageproc-software/annotations/YNP-BisonGraze/images"
OUTPUT_DIR = "/home/omartin9/wildimageproc-software/annotations/YNP-BisonGraze/wytrap_res"

summary = process_folder(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    species="wyoming_all",   # or 'wyoming_mammals', 'ynp_testbed', or a custom list
    device="auto",
    recursive=False,
    resume=True,
)
summary

[wytrap] 156 species in classifier list
[wytrap] loading detector (device=auto, threshold=0.2)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/omartin9/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


ValueError: Select a valid model version: MDV6-yolov9-c, MDV6-yolov9-e, MDV6-yolov10-c, MDV6-yolov10-e, MDV6-rtdetr-c

## Render boxes + labels for the first few results

In [ ]:
result_files = sorted(Path(OUTPUT_DIR).glob("*.json"))[:6]

for jf in result_files:
    rec = load_record(jf)
    if rec.error:
        print(f"{jf.name}: error {rec.error}")
        continue
    img = Image.open(rec.image_path).convert("RGB")
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(img)
    ax.set_title(jf.stem)
    for det in rec.detections:
        x1, y1, x2, y2 = det.box_xyxy
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            edgecolor="yellow", facecolor="none", linewidth=2,
        ))
        ax.text(x1, max(0, y1 - 6),
                f"{det.label} ({det.cls_score:.2f})",
                color="yellow", fontsize=10,
                bbox=dict(boxstyle="round", fc="black", ec="none", alpha=0.6))
    ax.axis("off")
    plt.show()

## CLI equivalent

From a shell on a GPU node:

```bash
wytrap detect \
    --input  /path/to/sample_images \
    --output /path/to/sample_results \
    --species wyoming_all \
    --device cuda --resume
```